# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaji412-lab/MY_ML_INTERNSHIP/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [22]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [23]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

BASE = "hf://datasets/FlyRank/internship-warehouse"

print("Connected!")

Connected!


**My rule**

Prioritize pages that have enough Google Search impressions but a low click-through rate (CTR). These pages already have search visibility but receive relatively few clicks, so they are candidates for a CTR review.

**Reason code**

* `LOW_CTR_OPPORTUNITY` — the page has sufficient search impressions but a low CTR.

**Action label**

* `CTR_REVIEW` — review the page's search result/title/description for possible CTR improvement.


In [24]:
import pandas as pd
import numpy as np

print("Pandas loaded successfully!")

Pandas loaded successfully!


In [25]:
# =========================
# W04 — Signal Checks
# =========================

import pandas as pd
import numpy as np

# CTR calculate
df_w04["ctr"] = np.where(
    df_w04["gsc_impressions"] > 0,
    df_w04["gsc_clicks"] / df_w04["gsc_impressions"],
    np.nan
)

# -------------------------
# Signal 1: CTR buckets
# -------------------------

# rank first so qcut can create 4 buckets even with many tied values
df_w04["ctr_bucket"] = pd.qcut(
    df_w04["ctr"].rank(method="first"),
    q=4,
    labels=["Q1_Low", "Q2", "Q3", "Q4_High"]
)

ctr_signal = (
    df_w04
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        avg_ctr=("ctr", "mean"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — CTR")
display(ctr_signal)


# -------------------------
# Signal 2: Impressions
# -------------------------

df_w04["impression_bucket"] = pd.qcut(
    df_w04["gsc_impressions"].rank(method="first"),
    q=4,
    labels=["Q1_Low", "Q2", "Q3", "Q4_High"]
)

impression_signal = (
    df_w04
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_ctr=("ctr", "mean"),
        avg_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — IMPRESSIONS")
display(impression_signal)

SIGNAL 1 — CTR


,ctr_bucket,n,avg_ctr,avg_impressions,avg_clicks
0,Q1_Low,902766,0.000000,47.690913,0.00000
1,Q2,902765,0.000000,43.844897,0.00000
2,Q3,902765,0.000000,45.022984,0.00000
3,Q4_High,902765,0.012323,174.327806,0.91035



SIGNAL 2 — IMPRESSIONS


,impression_bucket,n,avg_impressions,avg_ctr,avg_clicks
0,Q1_Low,902766,1.958793,0.004191,0.007072
1,Q2,902765,8.644960,0.002354,0.020350
2,Q3,902765,32.996132,0.002717,0.092349
3,Q4_High,902765,267.286766,0.003061,0.790580


**Verdict: CONFIRMED**

Higher CTR is associated with more clicks. Most pages in the lower CTR buckets have zero clicks, while the highest CTR bucket has a higher average click count. This supports using CTR as a useful signal for the baseline rule.

**Verdict: CONFIRMED**

Higher impressions are associated with more clicks. The highest-impression bucket has substantially more average clicks than the lower-impression buckets. This supports using impressions as a signal for prioritizing pages with greater search visibility.

In [26]:
# W04 — Real FlyRank flag: CTR vs Position

flag_check = df_w04[
    (df_w04["gsc_impressions"] > 0) &
    (df_w04["gsc_avg_position"].notna())
].copy()

flag_check["ctr"] = (
    flag_check["gsc_clicks"] / flag_check["gsc_impressions"]
)

# Position buckets
flag_check["position_bucket"] = pd.cut(
    flag_check["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21+"],
    include_lowest=True
)

ctr_position_signal = (
    flag_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        avg_ctr=("ctr", "mean"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

print("REAL FLYRANK FLAG — CTR vs POSITION")
display(ctr_position_signal)

REAL FLYRANK FLAG — CTR vs POSITION


,position_bucket,n,avg_ctr,avg_impressions,avg_clicks
0,Top 3,727362,0.004756,74.280199,0.282469
1,4-10,1456122,0.003473,94.655608,0.306175
2,11-20,519223,0.002770,56.596118,0.178053
3,21+,908354,0.001289,65.407183,0.085977


**Verdict: CONFIRMED**

CTR decreases as average search position gets worse. Pages ranking in the Top 3 have the highest average CTR (0.004756), while pages ranking at 21+ have the lowest average CTR (0.001289). This supports the CTR-vs-position FlyRank flag and confirms that CTR is a useful signal for prioritization.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [28]:
# =========================
# SECTION 2 — BASELINE QUEUE
# =========================

# Rank signals
df_w04["impression_rank"] = df_w04["gsc_impressions"].rank(pct=True)
df_w04["low_ctr_rank"] = 1 - df_w04["ctr"].rank(pct=True)

# Baseline score
df_w04["baseline_score"] = (
    0.5 * df_w04["impression_rank"] +
    0.5 * df_w04["low_ctr_rank"]
)

# Use the 25th percentile instead of median
ctr_threshold = df_w04["ctr"].quantile(0.25)

# Reason code
df_w04["reason_code"] = np.where(
    (df_w04["gsc_impressions"] > 0) &
    (df_w04["ctr"] <= ctr_threshold),
    "LOW_CTR_OPPORTUNITY",
    "OTHER"
)

# Action
df_w04["action"] = np.where(
    df_w04["reason_code"] == "LOW_CTR_OPPORTUNITY",
    "CTR_REVIEW",
    "NO_ACTION"
)

# Ranked queue
queue = (
    df_w04[
        [
            "client_hash_id",
            "content_hash_id",
            "report_date",
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

display(queue.head(10))

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,reason_code,action
0,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.0,8.613948,0.778937,LOW_CTR_OPPORTUNITY,CTR_REVIEW
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.0,0.181500,0.778936,LOW_CTR_OPPORTUNITY,CTR_REVIEW
2,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958,0,0.0,0.132532,0.778936,LOW_CTR_OPPORTUNITY,CTR_REVIEW
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,31472,0,0.0,0.083407,0.778935,LOW_CTR_OPPORTUNITY,CTR_REVIEW
4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,28973,0,0.0,0.000311,0.778935,LOW_CTR_OPPORTUNITY,CTR_REVIEW
5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,28947,0,0.0,0.002245,0.778935,LOW_CTR_OPPORTUNITY,CTR_REVIEW
6,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,27410,0,0.0,3.129405,0.778934,LOW_CTR_OPPORTUNITY,CTR_REVIEW
7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,24233,0,0.0,0.317996,0.778934,LOW_CTR_OPPORTUNITY,CTR_REVIEW
8,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,19301,0,0.0,2.261644,0.778932,LOW_CTR_OPPORTUNITY,CTR_REVIEW
9,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-24,17770,0,0.0,0.013056,0.778932,LOW_CTR_OPPORTUNITY,CTR_REVIEW


In [29]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved: work/outputs/baseline_action_score.csv")
print("Rows:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Rows: 3611061


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [31]:
# =========================
# SECTION 3 — TOP-20 REVIEW
# =========================

top20_review = queue.head(20).copy()

# Confidence note
top20_review["confidence_note"] = np.where(
    top20_review["reason_code"] == "LOW_CTR_OPPORTUNITY",
    "Moderate confidence: high priority based on impressions and low CTR.",
    "Low confidence: does not meet the low-CTR opportunity rule."
)

# What would make it wrong
top20_review["what_would_make_it_wrong"] = np.where(
    top20_review["reason_code"] == "LOW_CTR_OPPORTUNITY",
    "It could be wrong if tracking is incomplete, impressions are too low, or CTR is affected by search intent/position.",
    "It could be wrong if the CTR threshold is too strict or the page has an issue not captured by GSC data."
)

# Keep required review columns
top20_review = top20_review[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "baseline_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(top20_review)

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.0,8.613948,0.778937,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.0,0.181500,0.778936,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
2,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958,0,0.0,0.132532,0.778936,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,31472,0,0.0,0.083407,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,28973,0,0.0,0.000311,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,28947,0,0.0,0.002245,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
6,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,27410,0,0.0,3.129405,0.778934,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,24233,0,0.0,0.317996,0.778934,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
8,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,19301,0,0.0,2.261644,0.778932,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."
9,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-24,17770,0,0.0,0.013056,0.778932,CTR_REVIEW,LOW_CTR_OPPORTUNITY,Moderate confidence: high priority based on im...,"It could be wrong if tracking is incomplete, i..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [33]:
# =========================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# =========================

# Check the weakest-looking picks in the top 20
weak_picks = top20_review[
    (top20_review["gsc_impressions"] > 0) &
    (top20_review["gsc_clicks"] == 0)
].copy()

display(weak_picks[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "baseline_score",
        "action",
        "reason_code"
    ]
])

# Leakage check
# Confirm that no product flags or future-looking windows are being used.
product_flag_columns = [
    col for col in df_w04.columns
    if "product" in col.lower() or "flag" in col.lower()
]

future_columns = [
    col for col in df_w04.columns
    if any(word in col.lower() for word in ["future", "next", "lead", "conversion"])
]

print("Product/flag columns found:", product_flag_columns)
print("Future-looking columns found:", future_columns)

print("\nLeakage check:")
if len(product_flag_columns) == 0 and len(future_columns) == 0:
    print("PASS — No product flags or future-looking columns detected.")
else:
    print("REVIEW — Check the columns listed above before finalizing.")

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,action,reason_code
0,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,0,0.0,8.613948,0.778937,CTR_REVIEW,LOW_CTR_OPPORTUNITY
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,33383,0,0.0,0.181500,0.778936,CTR_REVIEW,LOW_CTR_OPPORTUNITY
2,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,32958,0,0.0,0.132532,0.778936,CTR_REVIEW,LOW_CTR_OPPORTUNITY
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,31472,0,0.0,0.083407,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY
4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,28973,0,0.0,0.000311,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY
5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,28947,0,0.0,0.002245,0.778935,CTR_REVIEW,LOW_CTR_OPPORTUNITY
6,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,27410,0,0.0,3.129405,0.778934,CTR_REVIEW,LOW_CTR_OPPORTUNITY
7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,24233,0,0.0,0.317996,0.778934,CTR_REVIEW,LOW_CTR_OPPORTUNITY
8,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,19301,0,0.0,2.261644,0.778932,CTR_REVIEW,LOW_CTR_OPPORTUNITY
9,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-24,17770,0,0.0,0.013056,0.778932,CTR_REVIEW,LOW_CTR_OPPORTUNITY


Product/flag columns found: []
Future-looking columns found: []

Leakage check:
PASS — No product flags or future-looking columns detected.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.